# Predial Vision MX — Phase 2: Temporal Multi-Date + Consensus
4 ESRI Wayback dates (2018-2024) + original + TTA + temporal consensus voting  
Target: F1 0.60 → 0.63-0.65

## 1. GPU Detection

In [ ]:
import os, sys, gc, time, json, gzip, math, warnings, urllib.request, concurrent.futures
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from urllib.request import urlretrieve
import subprocess

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large

import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize, shapes
from rasterio.crs import CRS
import shapely
from shapely.geometry import shape, mapping, box
import geopandas as gpd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
print('Libraries loaded successfully')

# === STRICT GPU GUARD — ABORT on P100 ===
if not torch.cuda.is_available():
    print('ERROR: No GPU detected. Aborting.')
    print('This notebook requires GPU T4 or better.')
    raise SystemExit('No GPU available')

props = torch.cuda.get_device_properties(0)
cc = props.major * 10 + props.minor
gpu_name = props.name
gpu_mem = props.total_memory / 1e9

if cc < 75:
    print(f'ERROR: GPU {gpu_name} has sm_{cc} (< sm_75)')
    print('P100 (sm_60) is INCOMPATIBLE with this notebook.')
    print('Please change Accelerator to GPU T4 x2 in Settings.')
    raise SystemExit(f'GPU {gpu_name} incompatible — need sm_75+')

print(f'GPU: {gpu_name} | CC: sm_{cc} | VRAM: {gpu_mem:.1f} GB')
DEVICE = 'cuda'
print(f'Device: {DEVICE}')

## 2. Configuration

In [ ]:
# Paths
WORKING_DIR = Path('/kaggle/working')
INPUT_DIR   = Path('/kaggle/input')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# MS Buildings download
MS_BUILDINGS_URL     = 'https://github.com/MarxCha/predial-vision-mx/releases/download/v0.2.0-data/ms_buildings_nr.geojson.gz'
MS_BUILDINGS_PATH    = WORKING_DIR / 'ms_buildings_nr.geojson.gz'
MS_BUILDINGS_GEOJSON = WORKING_DIR / 'ms_buildings_nr.geojson'

# Wayback release numbers (verified working)
WAYBACK_DATES = {
    '2024': 16453,
    '2022': 45134,
    '2020': 29260,
    '2018': 23448,
}

# Tile parameters (same as original mosaic)
ZOOM  = 17
X_MIN, X_MAX = 29360, 29396  # 37 tiles wide
Y_MIN, Y_MAX = 58241, 58271  # 31 tiles tall

# URL templates
WAYBACK_URL_TMPL = 'https://wayback.maptiles.arcgis.com/arcgis/rest/services/world_imagery/mapserver/tile/{release}/{z}/{y}/{x}'
ESRI_URL_TMPL    = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}'

# Model
CHIP_SIZE    = 256
STRIDE_TRAIN = 128
STRIDE_VAL   = 256
STRIDE_INFER = 128
BATCH_SIZE   = 16
NUM_EPOCHS   = 50
LEARNING_RATE = 1e-3
POS_WEIGHT    = 5.0
VAL_SPLIT     = 0.30
RANDOM_SEED   = 42

# Inference
THRESHOLDS     = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
PRED_THRESHOLD = 0.35

# Geometric filter
MIN_AREA_M2 = 20
MIN_RECT    = 0.45
MIN_COMPACT = 0.1

# Labels
BUILDINGS_COUNT = 0
LABEL_SOURCE    = 'MS Buildings (GitHub Release v0.2.0-data)'

# Output
MODEL_PATH       = WORKING_DIR / 'best_model_phase2.pth'
EVAL_PATH        = WORKING_DIR / 'phase2_eval.json'
GEOJSON_PATH     = WORKING_DIR / 'phase2_buildings.geojson'
CONSENSUS_PATH   = WORKING_DIR / 'temporal_consensus_prob.npz'
VIZ_PATH         = WORKING_DIR / 'phase2_visualization.png'
CURVES_PATH      = WORKING_DIR / 'phase2_training_curves.png'

print('Configuration set')
print(f'Wayback dates: {list(WAYBACK_DATES.keys())}')
print(f'Tile grid: X [{X_MIN},{X_MAX}] Y [{Y_MIN},{Y_MAX}] Z={ZOOM}')
print(f'Total tiles per date: {(X_MAX-X_MIN+1)*(Y_MAX-Y_MIN+1)}')

## 3. Download MS Buildings Labels

In [ ]:
def download_ms_buildings():
    if MS_BUILDINGS_GEOJSON.exists():
        print(f'MS Buildings GeoJSON already exists: {MS_BUILDINGS_GEOJSON}')
        return True
    if not MS_BUILDINGS_PATH.exists():
        print(f'Downloading MS Buildings from: {MS_BUILDINGS_URL}')
        try:
            urlretrieve(MS_BUILDINGS_URL, str(MS_BUILDINGS_PATH))
            print(f'Downloaded: {MS_BUILDINGS_PATH.stat().st_size / 1e6:.1f} MB')
        except Exception as e:
            print(f'Download failed: {e}')
            return False
    print('Decompressing...')
    with gzip.open(str(MS_BUILDINGS_PATH), 'rb') as f_in:
        with open(str(MS_BUILDINGS_GEOJSON), 'wb') as f_out:
            f_out.write(f_in.read())
    print(f'Decompressed: {MS_BUILDINGS_GEOJSON.stat().st_size / 1e6:.1f} MB')
    return True

ms_ok = download_ms_buildings()
print(f'MS Buildings ready: {ms_ok}')

## 4. Load Original Imagery from Kaggle Dataset

In [ ]:
def find_imagery():
    tif_files = list(INPUT_DIR.rglob('*.tif')) + list(INPUT_DIR.rglob('*.tiff'))
    tif_files = [f for f in tif_files if f.stat().st_size > 1e6]  # > 1MB
    print(f'Found {len(tif_files)} GeoTIFF file(s) in /kaggle/input')
    for f in tif_files:
        print(f'  {f}  ({f.stat().st_size/1e6:.1f} MB)')
    return tif_files

tif_files = find_imagery()

# Also check for nicolas-romero.tif specifically
nr_candidates = list(INPUT_DIR.rglob('nicolas-romero.tif')) + list(INPUT_DIR.rglob('nicolas_romero.tif'))
if nr_candidates:
    print(f'Found nicolas-romero.tif: {nr_candidates[0]}')

if not tif_files:
    print('No imagery found in /kaggle/input — generating synthetic tile for testing...')
    SYNTH_PATH = WORKING_DIR / 'synthetic_test.tif'
    west, south, east, north = -98.35, 26.00, -98.25, 26.08
    width, height = 512, 512
    transform = from_bounds(west, south, east, north, width, height)
    data = np.random.randint(50, 200, (3, height, width), dtype=np.uint8)
    with rasterio.open(
        str(SYNTH_PATH), 'w',
        driver='GTiff', height=height, width=width, count=3,
        dtype='uint8', crs=CRS.from_epsg(4326), transform=transform
    ) as dst:
        dst.write(data)
    tif_files = [SYNTH_PATH]
    print(f'Synthetic tile created: {SYNTH_PATH}')

IMAGE_PATH = max(tif_files, key=lambda f: f.stat().st_size)
print(f'\nUsing image: {IMAGE_PATH}')

with rasterio.open(str(IMAGE_PATH)) as src:
    IMG_CRS       = src.crs
    IMG_TRANSFORM = src.transform
    IMG_HEIGHT    = src.height
    IMG_WIDTH     = src.width
    IMG_BANDS     = src.count
    IMG_BOUNDS    = src.bounds
    print(f'  Size: {IMG_WIDTH} x {IMG_HEIGHT} px | Bands: {IMG_BANDS}')
    print(f'  CRS: {IMG_CRS}')
    print(f'  Bounds: {IMG_BOUNDS}')

## 5. Rasterize MS Buildings to Mask

In [ ]:
def rasterize_buildings(geojson_path, image_path, chunk_size=2048):
    """Chunked, memory-efficient rasterization of MS Buildings to match image grid."""
    global BUILDINGS_COUNT

    with rasterio.open(str(image_path)) as src:
        crs       = src.crs
        transform = src.transform
        height    = src.height
        width     = src.width
        bounds    = src.bounds

    print(f'Loading MS Buildings from {geojson_path}...')
    gdf = gpd.read_file(str(geojson_path))
    BUILDINGS_COUNT = len(gdf)
    print(f'  Total buildings in file: {BUILDINGS_COUNT:,}')

    if gdf.crs is None:
        gdf = gdf.set_crs('EPSG:4326')
    if gdf.crs != crs:
        print(f'  Reprojecting from {gdf.crs} to {crs}...')
        gdf = gdf.to_crs(crs)

    img_bbox = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
    gdf = gdf[gdf.geometry.intersects(img_bbox)].copy()
    BUILDINGS_COUNT = len(gdf)
    print(f'  Buildings within image bounds: {BUILDINGS_COUNT:,}')

    if BUILDINGS_COUNT == 0:
        print('  WARNING: No buildings overlap with image — using zero mask')
        return np.zeros((height, width), dtype=np.uint8)

    shapes_list = [(geom, 1) for geom in gdf.geometry if geom is not None and geom.is_valid]

    mask = np.zeros((height, width), dtype=np.uint8)
    n_chunks_y = math.ceil(height / chunk_size)
    n_chunks_x = math.ceil(width / chunk_size)
    total_chunks = n_chunks_y * n_chunks_x
    done = 0

    for cy in range(n_chunks_y):
        row_off = cy * chunk_size
        row_end = min(row_off + chunk_size, height)
        for cx in range(n_chunks_x):
            col_off = cx * chunk_size
            col_end = min(col_off + chunk_size, width)
            ch = row_end - row_off
            cw = col_end - col_off

            chunk_transform = rasterio.transform.from_origin(
                transform.c + col_off * transform.a,
                transform.f + row_off * transform.e,
                transform.a,
                -transform.e
            )

            chunk_mask = rasterize(
                shapes_list,
                out_shape=(ch, cw),
                transform=chunk_transform,
                fill=0,
                dtype=np.uint8,
                all_touched=True
            )
            mask[row_off:row_end, col_off:col_end] = chunk_mask
            done += 1
            if done % max(1, total_chunks // 10) == 0:
                print(f'  Rasterized chunk {done}/{total_chunks} ({100*done/total_chunks:.0f}%)')

    coverage = mask.mean() * 100
    print(f'  Mask coverage: {coverage:.2f}% positive pixels')
    return mask

if ms_ok and MS_BUILDINGS_GEOJSON.exists():
    MASK = rasterize_buildings(MS_BUILDINGS_GEOJSON, IMAGE_PATH)
else:
    print('MS Buildings not available — creating zero mask')
    MASK = np.zeros((IMG_HEIGHT, IMG_WIDTH), dtype=np.uint8)
    BUILDINGS_COUNT = 0

print(f'\nMask shape: {MASK.shape} | dtype: {MASK.dtype}')
print(f'Buildings used as labels: {BUILDINGS_COUNT:,}')

## 6. Download 4 Wayback Mosaics

In [ ]:
# Download pre-computed Wayback mosaics from GitHub Release (749MB total)
# These were pre-downloaded locally and uploaded as numpy arrays (7936x9472x3 uint8)

WAYBACK_RELEASE_URL = 'https://github.com/MarxCha/predial-vision-mx/releases/download/v0.3.0-wayback'

WAYBACK_PATHS = {}

for year in WAYBACK_DATES.keys():
    npz_path = WORKING_DIR / f'wayback_{year}.npz'
    
    if npz_path.exists() and npz_path.stat().st_size > 1e6:
        print(f'[{year}] Already downloaded: {npz_path.stat().st_size/1e6:.0f} MB')
        WAYBACK_PATHS[year] = str(npz_path)
        continue
    
    url = f'{WAYBACK_RELEASE_URL}/wayback_{year}.npz'
    print(f'[{year}] Downloading from GitHub Release...')
    try:
        urlretrieve(url, str(npz_path))
        size = npz_path.stat().st_size / 1e6
        print(f'[{year}] Downloaded: {size:.0f} MB')
        WAYBACK_PATHS[year] = str(npz_path)
    except Exception as e:
        print(f'[{year}] ERROR: {e}')
        WAYBACK_PATHS[year] = None

print('\nWayback download summary:')
for year, path in WAYBACK_PATHS.items():
    status = 'OK' if path and Path(path).exists() else 'FAILED'
    print(f'  {year}: {status}')

## 7. Dataset, Augmentations, and Training

In [ ]:
def read_image_full(image_path):
    """Read full image as HxWx3 uint8 numpy array."""
    with rasterio.open(str(image_path)) as src:
        bands = []
        for b in range(1, min(src.count + 1, 4)):
            band = src.read(b).astype(np.float32)
            p2, p98 = np.percentile(band[band > 0], [2, 98]) if band.any() else (0, 255)
            band = np.clip((band - p2) / max(p98 - p2, 1e-6) * 255, 0, 255)
            bands.append(band.astype(np.uint8))
        if len(bands) == 1:
            bands = bands * 3
        elif len(bands) == 2:
            bands.append(bands[0])
        img = np.stack(bands[:3], axis=2)  # HxWx3
    return img

print('Reading full image...')
IMG_ARRAY = read_image_full(IMAGE_PATH)
print(f'Image array shape: {IMG_ARRAY.shape} | dtype: {IMG_ARRAY.dtype}')
print(f'Mask shape: {MASK.shape}')

In [ ]:
def extract_chips(image, mask, chip_size, stride):
    H, W = image.shape[:2]
    chips_img, chips_msk = [], []
    for y in range(0, H - chip_size + 1, stride):
        for x in range(0, W - chip_size + 1, stride):
            chips_img.append(image[y:y+chip_size, x:x+chip_size])
            chips_msk.append(mask[y:y+chip_size, x:x+chip_size])
    return chips_img, chips_msk

print('Extracting training chips...')
all_imgs, all_msks = extract_chips(IMG_ARRAY, MASK, CHIP_SIZE, STRIDE_TRAIN)
print(f'Total chips (stride={STRIDE_TRAIN}): {len(all_imgs)}')

indices = list(range(len(all_imgs)))
train_idx, val_idx = train_test_split(indices, test_size=VAL_SPLIT, random_state=RANDOM_SEED)
print(f'Train: {len(train_idx)} | Val: {len(val_idx)}')

In [ ]:
class BuildingDataset(Dataset):
    def __init__(self, images, masks, transform=None):
        self.images    = images
        self.masks     = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img  = self.images[idx].copy()
        mask = self.masks[idx].copy().astype(np.float32)
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img  = augmented['image']
            mask = augmented['mask']
        return img, mask.unsqueeze(0) if isinstance(mask, torch.Tensor) else torch.from_numpy(mask).unsqueeze(0)

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.HueSaturationValue(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_aug = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

train_imgs = [all_imgs[i] for i in train_idx]
train_msks = [all_msks[i] for i in train_idx]
val_imgs   = [all_imgs[i] for i in val_idx]
val_msks   = [all_msks[i] for i in val_idx]

train_ds = BuildingDataset(train_imgs, train_msks, train_aug)
val_ds   = BuildingDataset(val_imgs,   val_msks,   val_aug)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, pos_weight=5.0):
        super().__init__()
        self.pos_weight = torch.tensor([pos_weight])
        self.bce_loss   = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))

    def forward(self, logits, targets):
        self.bce_loss.pos_weight = self.bce_loss.pos_weight.to(logits.device)
        bce = self.bce_loss(logits, targets)
        probs  = torch.sigmoid(logits)
        smooth = 1.0
        inter  = (probs * targets).sum()
        dice   = 1.0 - (2.0 * inter + smooth) / (probs.sum() + targets.sum() + smooth)
        return bce + dice

def build_model():
    model = deeplabv3_mobilenet_v3_large(weights='DEFAULT')
    model.classifier[4] = nn.Conv2d(256, 1, kernel_size=1)
    if hasattr(model, 'aux_classifier') and model.aux_classifier is not None:
        model.aux_classifier[4] = nn.Conv2d(10, 1, kernel_size=1)
    return model

print('Building model...')
model     = build_model().to(DEVICE)
criterion = DiceBCELoss(pos_weight=POS_WEIGHT)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')
print('Model ready')

In [ ]:
def compute_f1(pred, target, threshold=0.5):
    pred_bin = (pred > threshold).float()
    tp = (pred_bin * target).sum().item()
    fp = (pred_bin * (1 - target)).sum().item()
    fn = ((1 - pred_bin) * target).sum().item()
    prec = tp / (tp + fp + 1e-7)
    rec  = tp / (tp + fn + 1e-7)
    f1   = 2 * prec * rec / (prec + rec + 1e-7)
    return f1, prec, rec

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device).float()
        optimizer.zero_grad()
        out  = model(imgs)['out']
        out  = nn.functional.interpolate(out, size=masks.shape[-2:], mode='bilinear', align_corners=False)
        loss = criterion(out, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def val_epoch(model, loader, criterion, device, threshold=0.35):
    model.eval()
    total_loss, total_f1 = 0, 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device).float()
        out   = model(imgs)['out']
        out   = nn.functional.interpolate(out, size=masks.shape[-2:], mode='bilinear', align_corners=False)
        loss  = criterion(out, masks)
        probs = torch.sigmoid(out)
        f1, _, _ = compute_f1(probs, masks)
        total_loss += loss.item()
        total_f1   += f1
    return total_loss / len(loader), total_f1 / len(loader)

train_losses, val_losses, val_f1s = [], [], []
best_val_f1 = 0.0
best_epoch  = 0

print(f'Starting training: {NUM_EPOCHS} epochs | device: {DEVICE}')
print('-' * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    tr_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    vl_loss, vl_f1 = val_epoch(model, val_loader, criterion, DEVICE)

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    val_f1s.append(vl_f1)

    scheduler.step(vl_f1)

    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        best_epoch  = epoch
        torch.save(model.state_dict(), str(MODEL_PATH))

    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]['lr']
    if epoch % 5 == 0 or epoch == 1:
        marker = ' <-- best' if epoch == best_epoch else ''
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
              f'TrLoss: {tr_loss:.4f} | VlLoss: {vl_loss:.4f} | '
              f'VlF1: {vl_f1:.4f} | LR: {lr_now:.2e} | {elapsed:.1f}s{marker}')

print('-' * 60)
print(f'Best val F1: {best_val_f1:.4f} @ epoch {best_epoch}')
print(f'Model saved: {MODEL_PATH}')

In [ ]:
# Save training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss', color='steelblue')
axes[0].plot(val_losses,   label='Val Loss',   color='coral')
axes[0].axvline(best_epoch - 1, color='green', linestyle='--', alpha=0.7, label=f'Best epoch {best_epoch}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(val_f1s, label='Val F1', color='mediumseagreen')
axes[1].axvline(best_epoch - 1, color='green', linestyle='--', alpha=0.7, label=f'Best epoch {best_epoch}')
axes[1].axhline(best_val_f1, color='orange', linestyle=':', alpha=0.7, label=f'Best F1={best_val_f1:.4f}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 Score')
axes[1].set_title('Validation F1 Score'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Phase 2 Training — Predial Vision MX', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(CURVES_PATH), dpi=150, bbox_inches='tight')
plt.show()
print(f'Training curves saved: {CURVES_PATH}')

## 8. TTA Predict Chip Function

In [ ]:
@torch.no_grad()
def tta_predict_chip(model, chip, device):
    """chip: (1, C, H, W) tensor on device. Returns averaged probability (1, 1, H, W)."""
    preds = []

    # Original
    preds.append(torch.sigmoid(model(chip)['out']))

    # Horizontal flip
    f = torch.flip(chip, [3])
    preds.append(torch.flip(torch.sigmoid(model(f)['out']), [3]))

    # Vertical flip
    f = torch.flip(chip, [2])
    preds.append(torch.flip(torch.sigmoid(model(f)['out']), [2]))

    # Rotate 90
    r = torch.rot90(chip, 1, [2, 3])
    preds.append(torch.rot90(torch.sigmoid(model(r)['out']), 3, [2, 3]))

    # Rotate 180
    r = torch.rot90(chip, 2, [2, 3])
    preds.append(torch.rot90(torch.sigmoid(model(r)['out']), 2, [2, 3]))

    # Rotate 270
    r = torch.rot90(chip, 3, [2, 3])
    preds.append(torch.rot90(torch.sigmoid(model(r)['out']), 1, [2, 3]))

    return torch.stack(preds).mean(0)

print('TTA chip function defined — 6 augmentations per chip')

## 9. TTA Sliding Window Function

In [ ]:
infer_norm = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

def tta_sliding_window(model, imagery, device, chip_size=256, stride=128):
    """Run TTA inference over full image. imagery: CxHxW uint8. Returns probability map HxW float32."""
    # Convert CxHxW -> HxWxC for albumentations
    if imagery.ndim == 3 and imagery.shape[0] <= 4:
        img_hwc = np.transpose(imagery, (1, 2, 0))  # CxHxW -> HxWxC
    else:
        img_hwc = imagery  # already HxWxC

    # Ensure 3 channels
    if img_hwc.shape[2] == 1:
        img_hwc = np.repeat(img_hwc, 3, axis=2)
    elif img_hwc.shape[2] > 3:
        img_hwc = img_hwc[:, :, :3]

    H, W = img_hwc.shape[:2]
    pred_mask  = np.zeros((H, W), dtype=np.float32)
    count_mask = np.zeros((H, W), dtype=np.float32)

    model.eval()
    total = 0
    with torch.no_grad():
        for y in range(0, H - chip_size + 1, stride):
            for x in range(0, W - chip_size + 1, stride):
                chip = img_hwc[y:y+chip_size, x:x+chip_size]
                if chip.mean() < 0.02 * 255:
                    continue
                aug    = infer_norm(image=chip)
                chip_t = aug['image'].unsqueeze(0).to(device)  # (1, C, H, W)
                prob   = tta_predict_chip(model, chip_t, device)  # (1, 1, H, W)
                prob_np = prob[0, 0].cpu().numpy()  # (H, W)
                pred_mask[y:y+chip_size,  x:x+chip_size]  += prob_np
                count_mask[y:y+chip_size, x:x+chip_size] += 1
                total += 1
                if total % 500 == 0:
                    print(f'  {total} chips processed')

    count_mask[count_mask == 0] = 1
    return pred_mask / count_mask

print('TTA sliding window function defined')

## 10. TTA on Original Imagery

In [ ]:
# Load best model weights
print(f'Loading best model from {MODEL_PATH}...')
model.load_state_dict(torch.load(str(MODEL_PATH), map_location=DEVICE))
model.eval()
print('Best model loaded')

print('=== TTA on original imagery ===')
# IMG_ARRAY is HxWxC, convert to CxHxW for tta_sliding_window
imagery_chw = np.transpose(IMG_ARRAY, (2, 0, 1))  # HxWxC -> CxHxW
prob_original = tta_sliding_window(model, imagery_chw, DEVICE)
print(f'Original prob map: min={prob_original.min():.3f}, max={prob_original.max():.3f}, '
      f'mean={prob_original.mean():.3f}')

## 11. TTA on Each Wayback Date

In [ ]:
prob_maps = {'original': prob_original}

for year in WAYBACK_DATES.keys():
    npz_path = WAYBACK_PATHS.get(year)
    if npz_path is None or not Path(npz_path).exists():
        print(f'\n=== Wayback {year}: SKIPPED (not available) ===')
        continue

    print(f'\n=== TTA on Wayback {year} ===')
    # Load from npz (HxWx3 uint8)
    data = np.load(npz_path)
    wb_imagery_hwc = data['imagery']  # HxWx3
    print(f'Loaded: {wb_imagery_hwc.shape}')
    
    # Convert to CxHxW for tta_sliding_window
    wb_imagery_chw = np.transpose(wb_imagery_hwc, (2, 0, 1))  # HxWx3 -> CxHxW
    del wb_imagery_hwc, data
    gc.collect()

    # Handle size mismatch
    orig_h, orig_w = prob_original.shape
    wb_c, wb_h, wb_w = wb_imagery_chw.shape
    if wb_h != orig_h or wb_w != orig_w:
        min_h_tmp = min(orig_h, wb_h)
        min_w_tmp = min(orig_w, wb_w)
        print(f'  Size mismatch: original ({orig_h}x{orig_w}) vs wayback ({wb_h}x{wb_w})')
        print(f'  Cropping to: {min_h_tmp}x{min_w_tmp}')
        wb_imagery_chw = wb_imagery_chw[:, :min_h_tmp, :min_w_tmp]

    prob_maps[year] = tta_sliding_window(model, wb_imagery_chw, DEVICE)
    del wb_imagery_chw
    gc.collect()
    print(f'{year} prob map: min={prob_maps[year].min():.3f}, max={prob_maps[year].max():.3f}, '
          f'mean={prob_maps[year].mean():.3f}')

date_keys = [k for k in prob_maps.keys() if not k.startswith('_')]
print(f'\nProbability maps computed for: {date_keys}')

## 12. Temporal Consensus

In [ ]:
# Get crop dimensions if any wayback had size mismatch
crop_h = prob_maps.pop('_crop_h', None)
crop_w = prob_maps.pop('_crop_w', None)

# Align all maps to common size
date_keys = list(prob_maps.keys())
n_dates   = len(date_keys)

# Find minimum shape among all probability maps
min_h = min(prob_maps[k].shape[0] for k in date_keys)
min_w = min(prob_maps[k].shape[1] for k in date_keys)
print(f'Aligning {n_dates} probability maps to {min_h}x{min_w}...')

aligned = {k: prob_maps[k][:min_h, :min_w] for k in date_keys}

# Stack and compute consensus
all_probs = np.stack([aligned[k] for k in date_keys], axis=0)  # (N, H, W)
consensus_prob = all_probs.mean(axis=0)  # (H, W)

# Vote count: how many dates agree at threshold 0.5
vote_count = (all_probs > 0.5).sum(axis=0).astype(np.uint8)  # (H, W)

print(f'Consensus prob: min={consensus_prob.min():.3f}, max={consensus_prob.max():.3f}, '
      f'mean={consensus_prob.mean():.3f}')
print(f'Vote distribution ({n_dates} dates total):')
for v in range(n_dates + 1):
    count = int(np.sum(vote_count == v))
    print(f'  {v}/{n_dates} dates agree: {count:,} pixels ({100*count/vote_count.size:.1f}%)')

np.savez_compressed(
    str(CONSENSUS_PATH),
    consensus=consensus_prob,
    votes=vote_count,
    date_keys=np.array(date_keys)
)
print(f'\nConsensus saved: {CONSENSUS_PATH}')

# Also crop MASK to same size for evaluation
MASK_ALIGNED = MASK[:min_h, :min_w]
print(f'Mask aligned to {MASK_ALIGNED.shape}')

## 13. Threshold Sweep on Consensus

In [ ]:
def evaluate_at_threshold(pred, target, threshold):
    pred_bin = (pred > threshold).astype(np.float32)
    target_f = target.astype(np.float32)
    tp = float((pred_bin * target_f).sum())
    fp = float((pred_bin * (1 - target_f)).sum())
    fn = float(((1 - pred_bin) * target_f).sum())
    prec = tp / (tp + fp + 1e-7)
    rec  = tp / (tp + fn + 1e-7)
    f1   = 2 * prec * rec / (prec + rec + 1e-7)
    iou  = tp / (tp + fp + fn + 1e-7)
    return prec, rec, f1, iou

sweep_results = {}
print('Threshold sweep on CONSENSUS probability map vs MS Buildings mask:')
print(f'{"Threshold":>10} | {"Precision":>10} | {"Recall":>8} | {"F1":>8} | {"IoU":>8}')
print('-' * 56)

best_f1_consensus = 0.0
best_thresh_consensus = 0.35

for t in THRESHOLDS:
    p, r, f1, iou = evaluate_at_threshold(consensus_prob, MASK_ALIGNED, t)
    sweep_results[t] = {'precision': p, 'recall': r, 'f1': f1, 'iou': iou}
    marker = ' <-- best' if f1 > best_f1_consensus else ''
    if f1 > best_f1_consensus:
        best_f1_consensus     = f1
        best_thresh_consensus = t
    print(f'{t:>10.2f} | {p:>10.4f} | {r:>8.4f} | {f1:>8.4f} | {iou:>8.4f}{marker}')

print(f'\nBest consensus threshold: {best_thresh_consensus:.2f} (F1={best_f1_consensus:.4f})')

# Also run threshold sweep on original-only for comparison
print('\nThreshold sweep on ORIGINAL-ONLY probability map (baseline comparison):')
orig_aligned = aligned.get('original', prob_original[:min_h, :min_w])
sweep_orig   = {}
best_f1_orig = 0.0
best_thresh_orig = 0.35

for t in THRESHOLDS:
    p, r, f1, iou = evaluate_at_threshold(orig_aligned, MASK_ALIGNED, t)
    sweep_orig[t] = {'precision': p, 'recall': r, 'f1': f1, 'iou': iou}
    if f1 > best_f1_orig:
        best_f1_orig = f1
        best_thresh_orig = t
    print(f'{t:>10.2f} | {p:>10.4f} | {r:>8.4f} | {f1:>8.4f} | {iou:>8.4f}')

print(f'\nBest original threshold: {best_thresh_orig:.2f} (F1={best_f1_orig:.4f})')
improvement = best_f1_consensus - best_f1_orig
print(f'Consensus improvement: {improvement:+.4f}')

## 14. Vote-Based Thresholds

In [ ]:
mask_f = MASK_ALIGNED.astype(np.float32)

print(f'Vote-based thresholds (>= N/{n_dates} dates agree at 0.5):')
print(f'{"Min votes":>10} | {"Precision":>10} | {"Recall":>8} | {"F1":>8}')
print('-' * 44)

vote_results = {}
best_f1_vote = 0.0
best_min_votes = 1

for min_votes in range(1, n_dates + 1):
    binary = (vote_count >= min_votes).astype(np.float32)
    tp = float((binary * mask_f).sum())
    fp = float((binary * (1 - mask_f)).sum())
    fn = float(((1 - binary) * mask_f).sum())
    p  = tp / (tp + fp + 1e-8)
    r  = tp / (tp + fn + 1e-8)
    f1 = 2 * p * r / (p + r + 1e-8)
    vote_results[min_votes] = {'precision': p, 'recall': r, 'f1': f1}
    marker = ' <-- best' if f1 > best_f1_vote else ''
    if f1 > best_f1_vote:
        best_f1_vote  = f1
        best_min_votes = min_votes
    print(f'{min_votes:>10}/{n_dates}  | {p:>10.4f} | {r:>8.4f} | {f1:>8.4f}{marker}')

print(f'\nBest vote method: >= {best_min_votes}/{n_dates} votes (F1={best_f1_vote:.4f})')

# Decide overall best method
if best_f1_vote >= best_f1_consensus:
    BEST_METHOD   = f'vote_{best_min_votes}'
    BEST_F1_FINAL = best_f1_vote
    binary_best   = (vote_count >= best_min_votes).astype(np.uint8)
    print(f'\n>>> Best method: VOTE ({best_min_votes}/{n_dates}) — F1={BEST_F1_FINAL:.4f}')
else:
    BEST_METHOD   = f'consensus_{best_thresh_consensus}'
    BEST_F1_FINAL = best_f1_consensus
    binary_best   = (consensus_prob > best_thresh_consensus).astype(np.uint8)
    print(f'\n>>> Best method: CONSENSUS (thr={best_thresh_consensus}) — F1={BEST_F1_FINAL:.4f}')

## 15. Vectorize Best Prediction + Geometric Filter

In [ ]:
def rectangularity(poly):
    mbr = poly.minimum_rotated_rectangle
    return poly.area / mbr.area if mbr.area > 0 else 0.0

def compactness(poly):
    p = poly.length
    return (4 * np.pi * poly.area / (p * p)) if p > 0 else 0.0

def filter_building(poly, min_area_m2=20, min_rect=0.45, min_compact=0.1):
    area_m2 = poly.area * (111000 ** 2)
    if area_m2 < min_area_m2:
        return False
    if rectangularity(poly) < min_rect:
        return False
    if compactness(poly) < min_compact:
        return False
    return True

def vectorize_mask(binary_mask, transform, crs):
    polys = []
    for geom, val in shapes(binary_mask.astype(np.uint8), transform=transform):
        if val == 1:
            poly = shape(geom)
            if poly.is_valid and not poly.is_empty:
                polys.append(poly)
    return polys

print(f'Vectorizing best binary mask (method={BEST_METHOD})...')
raw_polys = vectorize_mask(binary_best, IMG_TRANSFORM, IMG_CRS)
print(f'Raw polygons before geometric filter: {len(raw_polys):,}')

# Apply geometric filter
before_count   = len(raw_polys)
filtered_polys = [p for p in raw_polys if filter_building(p, MIN_AREA_M2, MIN_RECT, MIN_COMPACT)]
after_count    = len(filtered_polys)
removed        = before_count - after_count

print(f'Before geometric filter: {before_count:,} polygons')
print(f'After geometric filter:  {after_count:,} polygons (removed {removed:,})')

## 16. Final Evaluation and Summary

In [ ]:
# Re-rasterize filtered polygons for final evaluation
print('Rasterizing filtered predictions for final evaluation...')

if filtered_polys:
    shapes_list = [(mapping(p), 1) for p in filtered_polys]
    final_pred_mask = rasterize(
        shapes_list,
        out_shape=(min_h, min_w),
        transform=IMG_TRANSFORM,
        fill=0, dtype=np.uint8, all_touched=True
    ).astype(np.float32)
else:
    final_pred_mask = np.zeros((min_h, min_w), dtype=np.float32)

final_p, final_r, final_f1, final_iou = evaluate_at_threshold(final_pred_mask, MASK_ALIGNED, 0.5)

# ============================================================
print('\n' + '=' * 56)
print('=== PHASE 2 RESULTS ===')  
print('=' * 56)
print(f'Dates used ({n_dates}): {date_keys}')
print()
print('CONSENSUS THRESHOLD SWEEP:')
print(f'{"Threshold":>10} | {"Precision":>10} | {"Recall":>8} | {"F1":>8}')
print('-' * 46)
for t in THRESHOLDS:
    r = sweep_results[t]
    marker = ' <-- best' if t == best_thresh_consensus else ''
    print(f'{t:>10.2f} | {r["precision"]:>10.4f} | {r["recall"]:>8.4f} | {r["f1"]:>8.4f}{marker}')
print()
print('VOTE SWEEP:')
for mv, vr in vote_results.items():
    marker = ' <-- best' if mv == best_min_votes else ''
    print(f'  >= {mv}/{n_dates} votes: P={vr["precision"]:.4f} R={vr["recall"]:.4f} F1={vr["f1"]:.4f}{marker}')
print()
print(f'ORIGINAL-ONLY baseline:  F1={best_f1_orig:.4f} (thr={best_thresh_orig})')
print(f'CONSENSUS best method:   F1={best_f1_consensus:.4f} (thr={best_thresh_consensus})')
print(f'VOTE best method:        F1={best_f1_vote:.4f} (>= {best_min_votes}/{n_dates})')
print(f'CHOSEN METHOD:           {BEST_METHOD}')
print()
print(f'Before geometric filter: {before_count:,} polygons')
print(f'After geometric filter:  {after_count:,} polygons (removed {removed:,})')
print()
print('FINAL METRICS (best method + geom filter):')
print(f'  Precision: {final_p:.4f}')
print(f'  Recall:    {final_r:.4f}')
print(f'  F1 Score:  {final_f1:.4f}')
print(f'  IoU:       {final_iou:.4f}')
print(f'  Polygons:  {after_count:,}')
print(f'  vs Baseline (original-only): {final_f1 - best_f1_orig:+.4f}')
print('=' * 56)

## 17. Save Outputs

In [ ]:
# Save phase2_eval.json
eval_data = {
    'phase': 2,
    'label_source': LABEL_SOURCE,
    'buildings_count': int(BUILDINGS_COUNT),
    'wayback_dates': {str(y): int(r) for y, r in WAYBACK_DATES.items()},
    'dates_processed': date_keys,
    'n_dates': n_dates,
    'consensus_threshold_sweep': {
        str(t): {k: float(v) for k, v in v2.items()}
        for t, v2 in sweep_results.items()
    },
    'vote_sweep': {
        str(mv): {k: float(v) for k, v in vr.items()}
        for mv, vr in vote_results.items()
    },
    'original_only_baseline': {
        'best_threshold': float(best_thresh_orig),
        'best_f1': float(best_f1_orig)
    },
    'best_method': BEST_METHOD,
    'best_f1_before_geom_filter': float(BEST_F1_FINAL),
    'before_geom_filter': int(before_count),
    'after_geom_filter':  int(after_count),
    'removed_by_filter':  int(removed),
    'final_metrics': {
        'precision': float(final_p),
        'recall':    float(final_r),
        'f1':        float(final_f1),
        'iou':       float(final_iou),
        'polygons':  int(after_count),
        'improvement_vs_original': float(final_f1 - best_f1_orig)
    },
    'training': {
        'epochs': NUM_EPOCHS,
        'best_epoch': int(best_epoch),
        'best_val_f1': float(best_val_f1),
        'batch_size': BATCH_SIZE,
        'chip_size': CHIP_SIZE,
        'stride_train': STRIDE_TRAIN,
        'tta_passes': 6
    },
    'geometry_filter': {
        'min_area_m2': MIN_AREA_M2,
        'min_rect':    MIN_RECT,
        'min_compact': MIN_COMPACT
    }
}

with open(str(EVAL_PATH), 'w') as f:
    json.dump(eval_data, f, indent=2)
print(f'eval.json saved: {EVAL_PATH}')

In [ ]:
# Save phase2_buildings.geojson
if filtered_polys:
    features = []
    for i, poly in enumerate(filtered_polys):
        area_m2 = poly.area * (111000 ** 2)
        features.append({
            'type': 'Feature',
            'geometry': mapping(poly),
            'properties': {
                'id':             i,
                'area_m2':        round(float(area_m2), 2),
                'rectangularity': round(float(rectangularity(poly)), 4),
                'compactness':    round(float(compactness(poly)), 4),
                'source':         f'phase2_{BEST_METHOD}'
            }
        })
    geojson_out = {
        'type': 'FeatureCollection',
        'crs': {'type': 'name', 'properties': {'name': str(IMG_CRS)}},
        'features': features
    }
    with open(str(GEOJSON_PATH), 'w') as f:
        json.dump(geojson_out, f)
    print(f'GeoJSON saved: {GEOJSON_PATH} ({len(features):,} buildings)')
else:
    print('No polygons to save')

In [ ]:
# Visualization: 5-panel (original + 4 wayback dates) + consensus
VIZ_CROP_SIZE = min(512, min_h, min_w)
cy = max(0, (min_h - VIZ_CROP_SIZE) // 2)
cx = max(0, (min_w - VIZ_CROP_SIZE) // 2)

img_crop  = IMG_ARRAY[:min_h, :min_w][cy:cy+VIZ_CROP_SIZE, cx:cx+VIZ_CROP_SIZE]
gt_crop   = MASK_ALIGNED[cy:cy+VIZ_CROP_SIZE, cx:cx+VIZ_CROP_SIZE]
cons_crop = consensus_prob[cy:cy+VIZ_CROP_SIZE, cx:cx+VIZ_CROP_SIZE]
vote_crop = vote_count[cy:cy+VIZ_CROP_SIZE, cx:cx+VIZ_CROP_SIZE]

# Get per-date probability crops for display
date_crops = {}
for k in date_keys:
    date_crops[k] = aligned[k][cy:cy+VIZ_CROP_SIZE, cx:cx+VIZ_CROP_SIZE]

# Build figure: 2 rows
# Row 1: Original RGB | GT Mask | Consensus Prob | Best Binary Pred | Vote Map
# Row 2: Probability maps per date
n_date_panels = len(date_keys)
fig, axes = plt.subplots(2, max(n_date_panels, 5), figsize=(5 * max(n_date_panels, 5), 10))

# Row 0
axes[0, 0].imshow(img_crop)
axes[0, 0].set_title('RGB Imagery', fontsize=11)
axes[0, 0].axis('off')

axes[0, 1].imshow(img_crop)
axes[0, 1].imshow(gt_crop, alpha=0.5, cmap='Reds', vmin=0, vmax=1)
axes[0, 1].set_title(f'GT Mask\n({BUILDINGS_COUNT:,} buildings)', fontsize=11)
axes[0, 1].axis('off')

im = axes[0, 2].imshow(cons_crop, cmap='viridis', vmin=0, vmax=1)
axes[0, 2].set_title('Consensus Prob Map', fontsize=11)
axes[0, 2].axis('off')
plt.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)

axes[0, 3].imshow(img_crop)
axes[0, 3].imshow(binary_best[cy:cy+VIZ_CROP_SIZE, cx:cx+VIZ_CROP_SIZE],
                   alpha=0.5, cmap='Blues', vmin=0, vmax=1)
axes[0, 3].set_title(f'Best Prediction\n{BEST_METHOD}\nF1={final_f1:.4f}', fontsize=11)
axes[0, 3].axis('off')

axes[0, 4].imshow(vote_crop, cmap='hot', vmin=0, vmax=n_dates)
axes[0, 4].set_title(f'Vote Map\n(0-{n_dates} dates agree)', fontsize=11)
axes[0, 4].axis('off')

# Hide extra axes in row 0
for i in range(5, axes.shape[1]):
    axes[0, i].axis('off')

# Row 1: per-date probability maps
for idx, k in enumerate(date_keys):
    if idx < axes.shape[1]:
        im2 = axes[1, idx].imshow(date_crops[k], cmap='viridis', vmin=0, vmax=1)
        axes[1, idx].set_title(f'{k} prob map', fontsize=11)
        axes[1, idx].axis('off')

# Hide remaining axes in row 1
for i in range(len(date_keys), axes.shape[1]):
    axes[1, i].axis('off')

red_patch  = mpatches.Patch(color='red',  alpha=0.6, label='Ground Truth')
blue_patch = mpatches.Patch(color='blue', alpha=0.6, label='Best Prediction')
fig.legend(handles=[red_patch, blue_patch], loc='lower center', ncol=2, fontsize=11)

plt.suptitle(
    f'Predial Vision MX — Phase 2: Temporal Consensus ({n_dates} dates)\n'
    f'Best method: {BEST_METHOD} | F1={final_f1:.4f} | P={final_p:.4f} | R={final_r:.4f} | Polygons={after_count:,}',
    fontsize=12, fontweight='bold'
)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(str(VIZ_PATH), dpi=150, bbox_inches='tight')
plt.show()
print(f'Visualization saved: {VIZ_PATH}')

In [ ]:
# Final summary
print('\n' + '=' * 56)
print('=== PHASE 2 OUTPUT FILES ===')
print('=' * 56)
output_files = [
    MODEL_PATH, EVAL_PATH, GEOJSON_PATH, CONSENSUS_PATH, VIZ_PATH, CURVES_PATH
]
for p in output_files:
    exists = Path(p).exists()
    size   = Path(p).stat().st_size / 1e6 if exists else 0
    status = f'{size:.2f} MB' if exists else 'MISSING'
    print(f'  {Path(p).name:45s} {status}')
print('=' * 56)
print()
print('PHASE 2 COMPLETE.')
print(f'  Dates used:  {date_keys}')
print(f'  Best method: {BEST_METHOD}')
print(f'  Final F1:    {final_f1:.4f}')
print(f'  Polygons:    {after_count:,}')